# 004 — Security Checks and Resource Limits

学习目标：

1. 理解静态安全检查的边界
2. 实现简化版危险模式检测
3. 用 `setrlimit()` 限制子进程资源
4. 理解环境变量清洁和路径安全
5. 对比 Clawith 真实实现

---


## 1. 静态安全检查——第一道防线

静态检查就是在代码执行**之前**，用字符串匹配发现危险模式。

Clawith 针对三种语言定义了不同等级的危险模式：


In [1]:
# Clawith 风格的静态安全检查（简化版）

ALWAYS_DANGEROUS_BASH = [
    "rm -rf /", "rm -rf ~", "sudo ", "mkfs", "dd if=",
    ":(){ :",  # fork bomb
    "chmod 777 /", "chown ", "shutdown", "reboot",
]

NETWORK_DANGEROUS_BASH = [
    "curl ", "wget ", "nc ", "ncat ", "ssh ", "scp ",
]

ALWAYS_DANGEROUS_PYTHON = [
    "subprocess", "shutil.rmtree", "os.system", "os.popen",
    "os.exec", "os.spawn",
]

NETWORK_DANGEROUS_PYTHON = [
    "socket", "http.client", "urllib.request", "requests",
    "ftplib", "smtplib", "telnetlib", "ctypes",
    "__import__", "importlib",
]


def check_code_safety(language: str, code: str, allow_network: bool = False) -> str | None:
    """安全检查。返回 None 表示安全，返回字符串表示被拦截的原因。"""
    code_lower = code.lower()

    if language == "bash":
        for pattern in ALWAYS_DANGEROUS_BASH:
            if pattern.lower() in code_lower:
                return f"Blocked: dangerous command ({pattern.strip()})"
        if not allow_network:
            for pattern in NETWORK_DANGEROUS_BASH:
                if pattern.lower() in code_lower:
                    return f"Blocked: network not allowed ({pattern.strip()})"
        if "../../" in code:
            return "Blocked: directory traversal"

    elif language == "python":
        for pattern in ALWAYS_DANGEROUS_PYTHON:
            if pattern.lower() in code_lower:
                return f"Blocked: dangerous operation ({pattern.strip()})"
        if not allow_network:
            for pattern in NETWORK_DANGEROUS_PYTHON:
                if pattern.lower() in code_lower:
                    return f"Blocked: network not allowed ({pattern.strip()})"

    return None  # 安全


# 测试
test_cases = [
    ("python", "print('hello')", True),
    ("python", "os.system('rm -rf /')", True),
    ("python", "import requests", False),
    ("python", "import requests", True),
    ("bash", "curl http://evil.com", False),
    ("bash", "curl http://evil.com", True),
    ("bash", "rm -rf /important", True),
]

for lang, code, net in test_cases:
    result = check_code_safety(lang, code, allow_network=net)
    status = "✅" if result is None else "❌"
    print(f"{status} lang={lang}, net={net}: {code[:40]:40s} → {result or 'OK'}")


✅ lang=python, net=True: print('hello')                           → OK
❌ lang=python, net=True: os.system('rm -rf /')                    → Blocked: dangerous operation (os.system)
❌ lang=python, net=False: import requests                          → Blocked: network not allowed (requests)
✅ lang=python, net=True: import requests                          → OK
❌ lang=bash, net=False: curl http://evil.com                     → Blocked: network not allowed (curl)
✅ lang=bash, net=True: curl http://evil.com                     → OK
❌ lang=bash, net=True: rm -rf /important                        → Blocked: dangerous command (rm -rf /)


### 静态检查的边界

静态检查是文本扫描，不是语义分析。它能挡住什么？


In [2]:
print("✅ 静态检查能挡住：")
print("  • rm -rf /  → 字符串匹配")
print("  • os.system('curl ...') → 包含 os.system")
print("  • import socket → 包含 socket")
print()
print("❌ 静态检查挡不住（需要运行时隔离）：")
print("  • os.system('r'+'m'+' -rf /')  → 字符串拼接绕过")
print("  • base64 编码后执行             → 编码绕过")
print("  • 从文件读取命令并 exec          → 间接执行")
print("  • Python bytecode 直接注入      → 非文本形态")
print()
print("所以静态检查只是第一道防线——不能替代 namespace 隔离。")


✅ 静态检查能挡住：
  • rm -rf /  → 字符串匹配
  • os.system('curl ...') → 包含 os.system
  • import socket → 包含 socket

❌ 静态检查挡不住（需要运行时隔离）：
  • os.system('r'+'m'+' -rf /')  → 字符串拼接绕过
  • base64 编码后执行             → 编码绕过
  • 从文件读取命令并 exec          → 间接执行
  • Python bytecode 直接注入      → 非文本形态

所以静态检查只是第一道防线——不能替代 namespace 隔离。


## 2. 资源限制——第二道防线

即使代码穿过了静态检查，它也不能为所欲为。  
Clawith 在子进程启动前用 `setrlimit()` 设好资源限额：


In [3]:
import resource

def apply_resource_limits(
    memory_mb: int = 256,
    cpu_seconds: int = 30,
    max_files: int = 64,
    max_processes: int = 32,
    max_file_size_mb: int = 10,
):
    """设置子进程资源限制。模仿 Clawith 的 _build_preexec_fn。"""
    limits = [
        (resource.RLIMIT_AS,     memory_mb * 1024 * 1024),       # 地址空间（内存）
        (resource.RLIMIT_CPU,    cpu_seconds),                    # CPU 时间
        (resource.RLIMIT_FSIZE,  max_file_size_mb * 1024 * 1024), # 文件写入大小
        (resource.RLIMIT_NOFILE, max_files),                      # 文件描述符
        (resource.RLIMIT_NPROC,  max_processes),                  # 子进程数
    ]
    if hasattr(resource, "RLIMIT_CORE"):
        limits.append((resource.RLIMIT_CORE, 0))  # 禁止 core dump

    for rsc, limit in limits:
        try:
            resource.setrlimit(rsc, (limit, limit))
        except Exception as e:
            print(f"  ⚠️ 无法设置 {rsc.name}: {e}")


print("资源限制设置完成（仅在本进程生效，不演示暴力测试）")
print()
print("Clawith 的限制值：")
print(f"  RLIMIT_AS     = 256 MB   # 防止内存溢出")
print(f"  RLIMIT_CPU    = timeout  # 防止死循环")
print(f"  RLIMIT_FSIZE  = 10 MB    # 防止磁盘写满")
print(f"  RLIMIT_NOFILE = 64       # 防止文件描述符耗尽")
print(f"  RLIMIT_NPROC  = 32       # 防止 fork bomb")
print(f"  RLIMIT_CORE   = 0        # 禁止 core dump")


资源限制设置完成（仅在本进程生效，不演示暴力测试）

Clawith 的限制值：
  RLIMIT_AS     = 256 MB   # 防止内存溢出
  RLIMIT_CPU    = timeout  # 防止死循环
  RLIMIT_FSIZE  = 10 MB    # 防止磁盘写满
  RLIMIT_NOFILE = 64       # 防止文件描述符耗尽
  RLIMIT_NPROC  = 32       # 防止 fork bomb
  RLIMIT_CORE   = 0        # 禁止 core dump


### 演示：fork bomb 被 RLIMIT_NPROC 阻止


In [4]:
# 理论演示：fork bomb 的工作原理和防御
print("Fork bomb 代码:  :(){ :|:& };:")
print()
print("执行流程:")
print("  1. 定义函数 :(){ ... }")
print("  2. 函数内递归调用自身，并通过管道启动后台进程")
print("  3. 每次调用创建 2 个新进程")
print("  4. 进程数呈指数增长: 1 → 2 → 4 → 8 → 16 → ...")
print()
print("防御手段:")
print("  RLIMIT_NPROC = 32")
print("  → 当子进程数达到 32 时，fork() 返回 EAGAIN")
print("  → bomb 无法继续扩散")
print("  → 系统不受影响")


Fork bomb 代码:  :(){ :|:& };:

执行流程:
  1. 定义函数 :(){ ... }
  2. 函数内递归调用自身，并通过管道启动后台进程
  3. 每次调用创建 2 个新进程
  4. 进程数呈指数增长: 1 → 2 → 4 → 8 → 16 → ...

防御手段:
  RLIMIT_NPROC = 32
  → 当子进程数达到 32 时，fork() 返回 EAGAIN
  → bomb 无法继续扩散
  → 系统不受影响


## 3. 环境变量清洁

子进程会继承父进程的环境变量。如果不加清理，可能导致信息泄露。


In [5]:
import os

def build_safe_env(work_path: str) -> dict:
    """构建清洁的环境变量。模仿 Clawith 的 _build_safe_env。"""
    return {
        "HOME": work_path,
        "PATH": "/usr/bin:/bin",
        "PYTHONDONTWRITEBYTECODE": "1",    # 不生成 .pyc
        "PYTHONNOUSERSITE": "1",           # 不使用用户 site-packages
        "TMPDIR": f"{work_path}/.tmp",
        "NODE_PATH": "",
        "BASH_ENV": "",
        "ENV": "",
    }


clean_env = build_safe_env("/workspace")
print("清洁后的环境变量（仅白名单条目）：")
for k, v in sorted(clean_env.items()):
    print(f"  {k:30s} = {v}")

print()
print(f"对比原始环境变量数量：被移除 {len(os.environ) - len(clean_env)} 个")
print(f"（包括 PATH、LD_PRELOAD、DBUS_SESSION_BUS_ADDRESS 等）")


清洁后的环境变量（仅白名单条目）：
  BASH_ENV                       = 
  ENV                            = 
  HOME                           = /workspace
  NODE_PATH                      = 
  PATH                           = /usr/bin:/bin
  PYTHONDONTWRITEBYTECODE        = 1
  PYTHONNOUSERSITE               = 1
  TMPDIR                         = /workspace/.tmp

对比原始环境变量数量：被移除 86 个
（包括 PATH、LD_PRELOAD、DBUS_SESSION_BUS_ADDRESS 等）


## 4. 路径安全

防止 Agent 通过 `../../etc/passwd` 逃逸到工作目录之外。


In [6]:
from pathlib import Path

def resolve_path_within_root(root: Path, rel_path: str = "") -> Path:
    """确保路径不会逃逸出 root 目录。"""
    root_resolved = root.resolve()
    candidate = Path(rel_path.strip())
    if candidate.is_absolute():
        raise ValueError("Absolute path not allowed")

    target = (root_resolved / candidate).resolve()
    try:
        target.relative_to(root_resolved)
    except ValueError:
        raise ValueError(f"Path escapes root: {rel_path}")
    return target


# 测试
workspace = Path("/tmp/test_workspace")
workspace.mkdir(parents=True, exist_ok=True)
(workspace / "safe_file.txt").write_text("hello")

test_cases = [
    ("safe_file.txt",       "✅ 允许"),
    ("subdir/../safe_file.txt", "✅ 允许（归一化后仍在 root 内）"),
    ("../../etc/passwd",    "❌ 拒绝：逃逸"),
    ("/etc/passwd",         "❌ 拒绝：绝对路径"),
]

for path, expected in test_cases:
    try:
        result = resolve_path_within_root(workspace, path)
        print(f"  {expected:20s} {path:30s} → {result}")
    except ValueError as e:
        print(f"  {expected:20s} {path:30s} → {e}")


  ✅ 允许                 safe_file.txt                  → /tmp/test_workspace/safe_file.txt
  ✅ 允许（归一化后仍在 root 内）  subdir/../safe_file.txt        → /tmp/test_workspace/safe_file.txt
  ❌ 拒绝：逃逸              ../../etc/passwd               → Path escapes root: ../../etc/passwd
  ❌ 拒绝：绝对路径            /etc/passwd                    → Absolute path not allowed


## 5. 对比 Clawith 真实实现

| 本 Notebook | Clawith 源码 | 位置 |
|---|---|---|
| `check_code_safety()` | `_check_code_safety()` | `subprocess_backend.py:47-91` |
| `apply_resource_limits()` | `_build_preexec_fn()` | `subprocess_backend.py:137-176` |
| `build_safe_env()` | `_build_safe_env()` | `subprocess_backend.py:124-135` |
| `resolve_path_within_root()` | `resolve_path_within_root()` | `workspace_paths.py:21-48` |

值得注意的差异：

- Clawith 的 `_build_preexec_fn` 还尝试了 `chroot`（以 root 运行时）
- `_check_code_safety` 对 Node.js 也有对应的检测模式
- 路径安全独立为一个模块 `workspace_paths.py`，不只被沙盒使用

---

**小结：** 静态扫描 + 资源限制 + 路径安全 + 环境清洁 = 沙盒的多层防御。  
下一份 Notebook 会把所有组件拼成完整系统。
